# Build ASEAN country baseline

Notebook ini membentuk `data/input/asean-country-baseline.csv` untuk prototype ARISE. Satu baris mewakili satu negara ASEAN yang memiliki data neraca beras AFSIS 2026 pada hasil analisis lokal.

Sumber langsung: `outputs_asean_clustering/asean_master_dataframe.csv`. File tersebut merupakan keluaran `ASEAN_Rice_Clustering_Analysis_added.ipynb`, yang menggabungkan proyeksi neraca beras AFSIS 2026 dan hasil clustering terpilih `k = 3`.

Notebook ini hanya mengekspor kolom baseline. Metrik turunan seperti self-sufficiency, production shortfall, dan stock-to-use dihitung untuk pemeriksaan, tetapi tidak disimpan agar aplikasi memiliki satu sumber perhitungan.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

PROJECT_DIR = Path.cwd()
SOURCE_CANDIDATES = [
    PROJECT_DIR / "outputs_asean_clustering" / "asean_master_dataframe.csv",
    PROJECT_DIR.parent / "outputs_asean_clustering" / "asean_master_dataframe.csv",
]
SOURCE_PATH = next((path for path in SOURCE_CANDIDATES if path.exists()), None)
if SOURCE_PATH is None:
    checked = "\n".join(f"- {path}" for path in SOURCE_CANDIDATES)
    raise FileNotFoundError(
        "asean_master_dataframe.csv tidak ditemukan. Path yang diperiksa:\n" + checked
    )

OUTPUT_PATH = PROJECT_DIR / "data" / "input" / "asean-country-baseline.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Source:", SOURCE_PATH)
print("Output:", OUTPUT_PATH)

## Output schema

- `iso3`: ISO 3166-1 alpha-3, dipakai sebagai kunci join aplikasi.
- `country_name`: nama negara yang konsisten dengan analisis lokal.
- `data_year`: tahun data domestik, yaitu 2026.
- `production_tonnes`: proyeksi produksi beras giling.
- `domestic_use_tonnes`: proyeksi penggunaan domestik.
- `beginning_stock_tonnes`: proyeksi stok awal.
- `profile_id`: cluster terpilih dari solusi `k = 3`.
- `simulator_enabled`: `True` jika produksi lebih kecil daripada penggunaan domestik.

In [ ]:
source = pd.read_csv(SOURCE_PATH)

required_columns = [
    "Country",
    "AFSIS_Year",
    "Production_ton",
    "Domestic_Utilization_ton",
    "Beginning_Stock_ton",
    "Selected_Cluster",
]
missing_columns = [column for column in required_columns if column not in source.columns]
if missing_columns:
    raise KeyError(f"Kolom sumber tidak lengkap: {missing_columns}")

print(f"Rows in master data: {len(source)}")
print(f"Columns in master data: {len(source.columns)}")
source[required_columns].head()

In [ ]:
ISO3_MAP = {
    "Brunei Darussalam": "BRN",
    "Cambodia": "KHM",
    "Indonesia": "IDN",
    "Lao PDR": "LAO",
    "Malaysia": "MYS",
    "Myanmar": "MMR",
    "Philippines": "PHL",
    "Singapore": "SGP",
    "Thailand": "THA",
    "Viet Nam": "VNM",
}

baseline_source = source.loc[
    source["AFSIS_Year"].eq(2026)
    & source[["Production_ton", "Domestic_Utilization_ton", "Beginning_Stock_ton"]]
        .notna().all(axis=1)
].copy()

baseline = pd.DataFrame({
    "iso3": baseline_source["Country"].map(ISO3_MAP),
    "country_name": baseline_source["Country"],
    "data_year": baseline_source["AFSIS_Year"].astype(int),
    "production_tonnes": baseline_source["Production_ton"].round().astype("int64"),
    "domestic_use_tonnes": baseline_source["Domestic_Utilization_ton"].round().astype("int64"),
    "beginning_stock_tonnes": baseline_source["Beginning_Stock_ton"].round().astype("int64"),
    "profile_id": baseline_source["Selected_Cluster"],
})
baseline["simulator_enabled"] = (
    baseline["production_tonnes"] < baseline["domestic_use_tonnes"]
)
baseline = baseline.sort_values("country_name").reset_index(drop=True)
baseline

## Validation

Pemeriksaan berikut memastikan bahwa baseline memiliki sepuluh negara dengan data AFSIS 2026, ISO3 unik, nilai volume non-negatif, dan empat negara defisit yang diaktifkan pada simulator. Timor-Leste tidak masuk output karena dataset lokal belum memiliki neraca AFSIS yang sebanding.

In [ ]:
assert len(baseline) == 10, f"Expected 10 AFSIS-covered countries, found {len(baseline)}"
assert baseline["iso3"].notna().all(), "Missing ISO3 mapping"
assert baseline["iso3"].is_unique, "ISO3 values must be unique"
assert baseline["country_name"].is_unique, "Country names must be unique"
assert baseline["data_year"].eq(2026).all(), "All rows must use the 2026 AFSIS horizon"
assert baseline[["production_tonnes", "domestic_use_tonnes", "beginning_stock_tonnes"]].ge(0).all().all(), "Volumes cannot be negative"
assert baseline["profile_id"].isin(["C1", "C2", "C3"]).all(), "Unexpected profile ID"

if "Cluster_k3" in baseline_source.columns:
    mismatches = baseline_source.loc[
        baseline_source["Selected_Cluster"] != baseline_source["Cluster_k3"],
        ["Country", "Selected_Cluster", "Cluster_k3"],
    ]
    assert mismatches.empty, "Selected_Cluster no longer matches the selected k=3 solution"

simulator_countries = baseline.loc[baseline["simulator_enabled"], "country_name"].tolist()
expected_simulator_countries = [
    "Brunei Darussalam", "Malaysia", "Philippines", "Singapore"
]
assert simulator_countries == expected_simulator_countries, (
    f"Unexpected simulator coverage: {simulator_countries}"
)

quality_check = baseline.assign(
    self_sufficiency_pct=lambda df: df["production_tonnes"] / df["domestic_use_tonnes"] * 100,
    production_shortfall_pct=lambda df: (
        100 - df["production_tonnes"] / df["domestic_use_tonnes"] * 100
    ).clip(lower=0),
    beginning_stock_to_use_pct=lambda df: (
        df["beginning_stock_tonnes"] / df["domestic_use_tonnes"] * 100
    ),
)

print("Validation passed.")
print("Simulator countries:", ", ".join(simulator_countries))
quality_check[[
    "country_name", "self_sufficiency_pct", "production_shortfall_pct",
    "beginning_stock_to_use_pct", "profile_id", "simulator_enabled"
]].round(2)

In [ ]:
baseline.to_csv(OUTPUT_PATH, index=False, encoding="utf-8", lineterminator="\n")

reloaded = pd.read_csv(OUTPUT_PATH)
assert reloaded.shape == baseline.shape, "Saved CSV shape differs from the baseline"
assert reloaded["iso3"].tolist() == baseline["iso3"].tolist(), "Saved row order changed"

print(f"Saved {len(reloaded)} rows to: {OUTPUT_PATH}")
reloaded

## Refresh workflow

1. Jalankan ulang `ASEAN_Rice_Clustering_Analysis_added.ipynb` jika data AFSIS atau clustering berubah.
2. Jalankan semua cell notebook ini.
3. Salin `data/input/asean-country-baseline.csv` ke folder data proyek ARISE jika website berada di repository yang berbeda.
4. Jangan mengubah metrik turunan secara manual. Hitung kembali dari tiga kolom volume pada script pembentuk data aplikasi.